**Mejora por cercania fisica en el teclado**


El sistema base utiliza la distancia de Levenshtein, que asume que todos los errores humanos son iguales (sustituir 'a' por 's' cuesta lo mismo que 'a' por 'z').

El objetivo de esta mejora es reflejar la realidad del error humano: los usuarios fallan por cercanía física en el teclado. 

Al penalizar menos los errores "lógicos", el autocorrector priorizará candidatos que tengan sentido sobre palabras aleatorias que simplemente están a la misma distancia matemática.

In [ ]:
from pathlib import Path
import re
import os

def normalizar_tildes(texto: str) -> str:
    return texto.translate(str.maketrans('áéíóúü', 'aeiouu'))

ruta_libros = Path('../libros')
files = os.listdir(ruta_libros)
words = {}
for file in files:
    with open(ruta_libros/file, 'r', encoding='UTF-8') as f:
        for linea in f:
            linea = normalizar_tildes(linea.lower())
            linea = re.sub(r'[^\w\s]','', linea)

            for word in linea.split():
                words[word] = words.get(word,0) + 1

**Funciones para aplicar una edicion**

Aplicamos las 4 funciones base, insertar, borrar, reemplazar e intercambiar:

In [2]:
def insert(word: str) -> set:
    alphabet = 'abcdefghijklmnñopqrstuvwxyz'
    words_created = set()
    for i in range(len(word) + 1):
        for char in alphabet:
            new_word = word[:i] + char + word[i:]
            words_created.add(new_word)
    
    return words_created

def delete(word: str) -> set:
    words_created = set()
    for i in range(len(word)):
        new_word = word[:i] + word[i+1:]
        words_created.add(new_word)

    return words_created
    
def replace(word: str) -> set:
    alphabet = 'abcdefghijklmnñopqrstuvwxyz'
    words_created = set()
    for i in range(len(word)):
        for char in alphabet:
            new_word = word[:i] + char + word[i+1:]
            words_created.add(new_word)

    return words_created

def exchange(word: str) -> set:
    words_created = set()
    for i in range(len(word) - 1):
        new_word = word[:i] + word[i+1] + word[i] + word[i+2:]
        words_created.add(new_word)

    return words_created

**Funcion una edicion y dos ediciones**

Debemos recibir una palabra y aplicar las 4 operaciones basicas y devolver las palabras unicas a distancia uno o distancia dos:

In [3]:
def una_edicion(word:str, intercambiar: bool=False) ->set:
    return insert(word).union(delete(word), replace(word), exchange(word) if intercambiar else set())

def dos_ediciones(word:str,intercambiar:bool =False) ->set:
    palabras_ed1 = una_edicion(word, intercambiar)
    palabras_ed2 = set()
    for palabra in palabras_ed1:
        palabras_ed2 = palabras_ed2.union(una_edicion(palabra, intercambiar))
    return palabras_ed2

**Filtrado de candidatos por vocabulario**

A partir de los candidatos a una y dos ediciones, nos quedamos solo con los que existen en el corpus para evitar sugerencias imposibles.

In [4]:
def palabras_conocidas(candidatas: set, vocabulario: dict, min_freq: int = 3) -> set:
    return {pal for pal in candidatas if vocabulario.get(pal, 0) >= min_freq}

#De esta forma no tenemos que calcular las dos ediciones habiendo ya calculado palabras a una edicion
def candidatos(word: str, vocabulario: dict, intercambiar: bool = True, min_freq: int = 3) -> set:
    if vocabulario.get(word, 0) >= min_freq:
        return {word}
    c1 = palabras_conocidas(una_edicion(word, intercambiar), vocabulario, min_freq=min_freq)
    if c1:
        return c1
    c2 = palabras_conocidas(dos_ediciones(word, intercambiar), vocabulario, min_freq=min_freq)
    if c2:
        return c2
    return {word}

**Distancia de teclado + Damerau-Levenshtein ponderada**

Para reemplazos, la penalización depende de qué tan cerca estén las teclas. Sustituir letras vecinas cuesta menos que sustituir letras lejanas.
Además para incluir una mejora con poco cambio en la arquitectura general, agregamos un hiper parametro como lo es min_freq el cual permite que 
haya un numero minimo de veces que aparezca una palabra para que asi en teoria, sea mas problable la palabra corregida y no caiga en sesgos por el 
conjunto de entrenamiento.

In [5]:
import numpy as np

def mapa_teclado_es() -> dict:
    filas = ["qwertyuiop", "asdfghjklñ", "zxcvbnm"]
    posiciones = {}
    for i, fila in enumerate(filas):
        for j, letra in enumerate(fila):
            posiciones[letra] = (i, j)
    return posiciones

TECLADO_POS = mapa_teclado_es() #cargamos las posiciones de las letras

def distancia_euclidea_2d(p1: np.ndarray, p2: np.ndarray) -> float:
    pos1 = np.array(p1)
    pos2 = np.array(p2)
    return np.sqrt(np.sum((pos1-pos2)**2))

def costo_reemplazo(letra_origen: str, letra_destino: str, a: float = 2.0) -> float:
    # si no cambia la letra, no hay costo
    if letra_origen == letra_destino:
        return 0.0

    pos_origen = TECLADO_POS.get(letra_origen)
    pos_destino = TECLADO_POS.get(letra_destino)

    # si alguna letra no está en el mapa, usamos costo estándar
    if pos_origen is None or pos_destino is None:
        return 1.0

    # distancia euclídea normalizada para no desbalancear inserción/borrado
    distancia = distancia_euclidea_2d(pos_origen, pos_destino)
    return distancia / a

def distancia_levenshtein_ponderada(origen: str, destino: str, a: float=2.0, costo_intercambio: float = 0.5) -> float:
    filas,columnas = len(origen) + 1, len(destino) + 1

    # matriz = costo mínimo para transformar palabra origen en palabra destino
    matriz = np.zeros(shape=(filas, columnas), dtype=np.float32)

    # caso base: convertir a cadena vacía (borrados) o desde vacía (inserciones)
    matriz[:, 0] = np.arange(filas).ravel()
    matriz[0, :] = np.arange(columnas).ravel()

    for i in range(1, filas):
        for j in range(1, columnas):
            costo_insertar = matriz[i][j - 1] + 1.0
            costo_borrar = matriz[i - 1][j] + 1.0
            costo_reemplazar_actual = matriz[i - 1][j - 1] + costo_reemplazo(origen[i - 1], destino[j - 1], a=a)
            matriz[i][j] = min(costo_insertar, costo_borrar, costo_reemplazar_actual)

            if (i > 1 and j > 1 and origen[i - 1] == destino[j - 2] and origen[i - 2] == destino[j - 1]):
                matriz[i][j] = min(matriz[i][j], matriz[i - 2][j - 2] + costo_intercambio)

    return matriz[filas-1][columnas-1]

**Ranking final de sugerencias**

Ordenamos por menor distancia y, en empate, por mayor frecuencia en el corpus.

In [6]:
def sugerencias(word: str, corpus: dict, top_k: int = 10, intercambiar: bool = True, a: float = 4.0, min_freq: int = 3, costo_intercambio: float = 0.5):
    word = normalizar_tildes(word.lower())
    cands = candidatos(word, corpus, intercambiar, min_freq=min_freq)

    ranking = []
    for cand in cands:
        dist = distancia_levenshtein_ponderada(word, cand, a=a, costo_intercambio=costo_intercambio)
        freq = corpus.get(cand, 0)
        ranking.append((cand, dist, freq))

    ranking.sort(key=lambda x: (x[1], -x[2], x[0]))  # (dist, freq desc, palabra)
    return ranking[:top_k]


**Pruebas con el conjunto de validacion**

In [7]:
# X_val: 20 frases del corpus para validacion.
# Esta seleccion esta pensada para comparar una configuracion base
# (por ejemplo a=1, min_freq=1) frente a la version mejorada.
X_val = [
    ("en los gobiernos civiles por donde paso el matrimonio no quedaron ni los clavos", "en los gobiernos civiles por donde paso el matrimonio no quedaron ni los clvos"),
    ("no vas ahora a vernos le dijo alguna vez que le encontro en la calle catalina", "no vas ahora a vrnos le dijo alguna vez que le encontro en la calle catalina"),
    ("no creo que habras venido a contarme unas cuantas bolas", "no creo que habras veido a contarme unas cuantas bolas"),
    ("esta ahora con unos amigos en el comedor", "esta ahora con unos aimgos en el comedor"),
    ("aqui no corre tanto el aire dijo un viejo mendigo que se preparaba a tenderse cerca de manuel", "aqui no corre tanto el aire dijo un viejo menigo que se preparaba a tenderse cerca de manuel"),
    ("tome posesion de mi cuchitril y comence mis trabajos", "tome posesion de mi cuchitril y comence mis trabjos"),
    ("ochoa grita ya estan ahi", "ochoa girta ya estan ahi"),
    ("comimos el arroz que estaba excesivamente sabroso", "comimos el ardoz que estaba excesivamente sabroso"),
    ("usted siempre podra llevar una buena vida si quiere replico mingote", "usted siempre opdra llevar una buena vida si quiere replico mingote"),
    ("durante los demas dias de la semana la barraca del domador estuvo vacia", "durante los demas dias de la semana la barraca del domador estuvo avcia"),
    ("la petra y el no se entendian y el matrimonio andaba siempre a trastazos", "la petra y el no se entendian y el matrimonio nadaba siempre a trastazos"),
    ("el chico no podra ir al colegio", "el chico no podar ir al colegio"),
    ("tengo que enviar todos los meses duros a mi familia para que mi madre y mis dos hermanas vayan viviendo", "tengo que enviar todos los meses duros a mi familia para que mi madre y mis dos hermanas vagan viviendo"),
    ("comenzaba a hacer calor pasaban muchos curas por las callejuelas", "comenzaba a hacer calor pasaban muchos cruas por las callejuelas"),
    ("al amanecer del dia mina con la columna en orden de combate entro en el pueblo", "al amanecer del dia mina con la columna en orden de combat entro en el pueblo"),
    ("me ha recordado usted a su excelencia el duque de otranto", "me ha recordado usted a su excelencia el duqe de otranto"),
    ("el toque de oracion el del angelus el de la agonia el de la misa el de los funerales", "el yoque de oracion el del angelus el de la agonia el de la misa el de los funerales"),
    ("pero no crea usted que todos tienen un gran respeto ni por don carlos ni por sus generales", "pero no crea usted que todos tienen un gran respeto ni por don carlos ni por sus generalse"),
    ("riego impaciente mando cinco compañias y ordeno la prision inmediata de los generales", "riego impaciente mando cinco compañias y ordeno la prision inmediata de los generalse"),
    ("el pasado la feria de los discretos", "el pasado la fedia de los discretos"),
]


In [8]:
def corregir_palabra(word: str, corpus: dict, intercambiar: bool = True, a: float = 4.0, min_freq: int = 3, costo_intercambio: float = 0.5) -> str:
    ranking = sugerencias(word, corpus, top_k=1, intercambiar=intercambiar, a=a, min_freq=min_freq, costo_intercambio=costo_intercambio)
    if not ranking:
        return word
    return ranking[0][0]


def corregir_frase(frase: str, corpus: dict, intercambiar: bool = True, a: float = 4.0, min_freq: int = 3, costo_intercambio: float = 0.5) -> str:
    tokens = normalizar_tildes(frase.lower()).split()
    corregidos = [corregir_palabra(tok, corpus, intercambiar=intercambiar, a=a, min_freq=min_freq, costo_intercambio=costo_intercambio) for tok in tokens]
    return " ".join(corregidos)


def medir_accuracy(X_val: list, corpus: dict, intercambiar: bool = True, a: float = 4.0, min_freq: int = 3, costo_intercambio: float = 0.5, devolver_detalles: bool = True) -> dict:
    total = len(X_val)
    aciertos = 0
    detalles = []

    for frase_correcta, frase_con_error in X_val:
        frase_correcta_norm = normalizar_tildes(frase_correcta.lower())
        pred = corregir_frase(frase_con_error, corpus, intercambiar=intercambiar, a=a, min_freq=min_freq)
        ok = pred == frase_correcta_norm
        aciertos += int(ok)

        if devolver_detalles:
            detalles.append({
                "frase_correcta": frase_correcta_norm,
                "frase_con_error": normalizar_tildes(frase_con_error.lower()),
                "prediccion": pred,
                "acierto": ok,
            })

    resultado = {"accuracy": aciertos / total if total > 0 else 0.0, "aciertos": aciertos, "total": total}

    if devolver_detalles:
        resultado["detalles"] = detalles

    return resultado


base = medir_accuracy(X_val, words, intercambiar=True, a=1.0, min_freq=1, costo_intercambio=0.5, devolver_detalles=False)
print(f"Base -> {base['accuracy']:.2%} ({base['aciertos']}/{base['total']})")

mejor = None
for a in [1.0, 2.0, 1.5, 3.0, 4.0]:
    for min_freq in [1,2,3,4,5]:
        for costo_intercambio in [0.1, 0.2, 0.25, 0.5, 1]:
            r = medir_accuracy(X_val, words, intercambiar=True, a=a, min_freq=min_freq, costo_intercambio=costo_intercambio, devolver_detalles=False)
            print(f"a={a}, min_freq={min_freq}, costo_intercambio={costo_intercambio} -> {r['accuracy']:.2%} ({r['aciertos']}/{r['total']})")
            if mejor is None or r['accuracy'] > mejor['resultado']['accuracy']:
                mejor = {"a": a, "min_freq": min_freq, "resultado": r, "costo_intercambio": costo_intercambio}

print()
print(
    f"Mejor -> a={mejor['a']}, min_freq={mejor['min_freq']}, costo_intercambio={mejor['costo_intercambio']} | "
    f"{mejor['resultado']['accuracy']:.2%} ({mejor['resultado']['aciertos']}/{mejor['resultado']['total']})"
)

res = medir_accuracy(X_val, words, intercambiar=True, a=mejor['a'], min_freq=mejor['min_freq'])
print(f"Seleccionado para detalle -> {res['accuracy']:.2%} ({res['aciertos']}/{res['total']})")
print()


Base -> 60.00% (12/20)
a=1.0, min_freq=1, costo_intercambio=0.1 -> 60.00% (12/20)
a=1.0, min_freq=1, costo_intercambio=0.2 -> 60.00% (12/20)
a=1.0, min_freq=1, costo_intercambio=0.25 -> 60.00% (12/20)
a=1.0, min_freq=1, costo_intercambio=0.5 -> 60.00% (12/20)
a=1.0, min_freq=1, costo_intercambio=1 -> 60.00% (12/20)
a=1.0, min_freq=2, costo_intercambio=0.1 -> 85.00% (17/20)
a=1.0, min_freq=2, costo_intercambio=0.2 -> 85.00% (17/20)
a=1.0, min_freq=2, costo_intercambio=0.25 -> 85.00% (17/20)
a=1.0, min_freq=2, costo_intercambio=0.5 -> 85.00% (17/20)
a=1.0, min_freq=2, costo_intercambio=1 -> 85.00% (17/20)
a=1.0, min_freq=3, costo_intercambio=0.1 -> 90.00% (18/20)
a=1.0, min_freq=3, costo_intercambio=0.2 -> 90.00% (18/20)
a=1.0, min_freq=3, costo_intercambio=0.25 -> 90.00% (18/20)
a=1.0, min_freq=3, costo_intercambio=0.5 -> 90.00% (18/20)
a=1.0, min_freq=3, costo_intercambio=1 -> 90.00% (18/20)
a=1.0, min_freq=4, costo_intercambio=0.1 -> 80.00% (16/20)
a=1.0, min_freq=4, costo_intercambio

Comprobamos que hay una mejora notoria frente al autocorrector base, el cual simulamos con a=1.0, min_freq=1 y costo_intercambio=0.5.

De todos los hiper parametros probados en el conjunto de validacion, la mejor configuracion fue a=2.0, min_freq=3 y costo_intercambio=0.1, con un accuracy de 95.00% (19/20).

En el proximo bloque de codigo, mostramos el detalle frase por frase de esa mejor combinacion de hiperparametros:

In [9]:
res_mejor = medir_accuracy(X_val, words, intercambiar=True, a=mejor['a'], min_freq=mejor['min_freq'], costo_intercambio=mejor['costo_intercambio'])
print(f"Frases con la mejor combinacion de hiperparametros: a={mejor['a']}, min_freq={mejor['min_freq']}, costo_intercambio={mejor['costo_intercambio']}")
print()

# detalle frase por frase
for i, item in enumerate(res_mejor.get("detalles", []), start=1):
    estado = "OK" if item["acierto"] else "FAIL"
    print(f"[{i:02d}] {estado}")
    print(f"  error:      {item['frase_con_error']}")
    print(f"  prediccion: {item['prediccion']}")
    print(f"  correcta:   {item['frase_correcta']}")
    print()


Frases con la mejor combinacion de hiperparametros: a=2.0, min_freq=3, costo_intercambio=0.1

[01] OK
  error:      en los gobiernos civiles por donde paso el matrimonio no quedaron ni los clvos
  prediccion: en los gobiernos civiles por donde paso el matrimonio no quedaron ni los clavos
  correcta:   en los gobiernos civiles por donde paso el matrimonio no quedaron ni los clavos

[02] OK
  error:      no vas ahora a vrnos le dijo alguna vez que le encontro en la calle catalina
  prediccion: no vas ahora a vernos le dijo alguna vez que le encontro en la calle catalina
  correcta:   no vas ahora a vernos le dijo alguna vez que le encontro en la calle catalina

[03] OK
  error:      no creo que habras veido a contarme unas cuantas bolas
  prediccion: no creo que habras venido a contarme unas cuantas bolas
  correcta:   no creo que habras venido a contarme unas cuantas bolas

[04] OK
  error:      esta ahora con unos aimgos en el comedor
  prediccion: esta ahora con unos amigos en el come

In [11]:
print(words['coro'])
print(words['claro'])

print()

print(words['astroso'])
print(words['sabroso'])

print()

print(words['vagan'])
print(words['vaga'])
print(words['vayan'])

36
228

3
1

1
37
30


**Analisis de la mejor configuracion**

---

**Caso 1**

---
error:      el niño tiene el pelo muy clro

prediccion: el niño tiene el pelo muy claro
  
correcta:   el niño tiene el pelo muy claro

Este es un ejemplo donde la configuracion elegida funciona bien. El error es un borrado simple y la palabra correcta sigue quedando bien posicionada en el ranking por la combinacion de distancia y frecuencia.

---

**Caso 2**

---
error:      comimos el ardoz que estaba excesivamente sabroso

prediccion: comimos el arroz que estaba excesivamente astroso

correcta:   comimos el arroz que estaba excesivamente sabroso


Aqui aparece el unico fallo de la mejor configuracion del conjunto de validacion. La palabra con error, "ardoz", si se corrige bien a "arroz".


El problema ocurre con "sabroso": como min_freq=3, una palabra correcta pero poco frecuente puede dejar de considerarse valida y el sistema intenta reemplazarla por otra palabra cercana que si supere el umbral de frecuencia.

Este caso muestra el principal trade-off del modelo: min_freq mejora mucho el resultado global, pero puede sobrecorregir palabras correctas y raras.

---

**Caso 3**

---

error:      tengo que enviar todos los meses duros a mi familia para que mi madre y mis dos hermanas vagan viviendo

prediccion: tengo que enviar todos los meses duros a mi familia para que mi madre y mis dos hermanas vayan viviendo

correcta:   tengo que enviar todos los meses duros a mi familia para que mi madre y mis dos hermanas vayan viviendo


Este caso sirve para mostrar una mejora frente a configuraciones mas debiles. Con la configuracion final, "vagan" si se corrige a "vayan".

Al subir a=2.0, el costo de reemplazo por cercania fisica queda mejor balanceado y ya no penaliza tanto este tipo de sustitucion. Junto con el filtro por frecuencia, eso hace que "vayan" quede por encima de alternativas peores.


**Conclusion**

Las mejoras implementadas si superan al autocorrector base en este conjunto de validacion. El baseline con a=1.0, min_freq=1 y costo_intercambio=0.5 obtiene 60.00% (12/20), mientras que la mejor configuracion obtiene 95.00% (19/20) con a=2.0, min_freq=3 y costo_intercambio=0.1.

La mejora no viene de un solo cambio: min_freq limpia palabras poco confiables del corpus, la distancia ponderada por cercania de teclado mejora el ranking de reemplazos plausibles y Damerau-Levenshtein incorpora el intercambio como un error barato. El unico fallo restante muestra el costo de esa decision: al exigir una frecuencia minima, algunas palabras correctas pero raras pueden ser sobrecorregidas.